FINE-TUNING DI LLM SU DATASET PERSONALIZZATI

Un modello base è un eccezionale completatore statistico di frasi, ma spesso non sa come rispondere a una domanda diretta o eseguire un compito specifico.
Non ha ancora capito il concetto di domanda e riposta.
L'Instruction Tuning è il processo di Supervised Fine-Tuning (SFT) che addestra il modello su coppie di istruzioni e risposta per allineare il comportamento alle aspettative umane.
Trasformiamo il pappagallo stocastico in un assistente supervisoinato.
Per capire che dopo il comando deve esserci un'esecuzione.

Prendo un modello già addestrato, esempio Qwen, e continuo l'addestramento usando esempi specifici del mio dominio, così da modificare il comportamento.
Il fine-tuning di un LLM con dataset personalizzato, risponde alla domanda:
"come faccio a trasformare un Qwen (o altro LLM) generico in un Qwen specializzato nell'interpretazione degli ordini (esempio), email e documenti commerciali ERP?"
Questa è esattamente una situazione da fine-tuning, esattamente di Supervised Fine-Tuning, SFT.
Qwen preaddestrato -> dataset personalizzato con esempi corretti -> fine-tuning -> Qwen personalizzato.
Non do al modello solo i documenti da leggere, ma do esempi/input (documenti da leggere) e output corretto.
Durante il training il modello modifica i propri parametri per aumentare la probabilità di produrre output simili in futuro.
Hugging-Face considera l'SFT uno dei principali metodi post-training.

Stai facendo apprendimento supervisionato, devi avere pertanto x (input) ed y (output)

Il codice del training è relativamente semplice, il dataset è la parte difficile.
Un cattivo dataset produce un cattivo modello anche con una pipeline perfetta.

Supervised Fine-Tuning (SFT)
Usiamo un Prompt Template: una struttura rigidta che dice al modello da qui inizia il comando e qui finisce, separe chiaramente l'istruzione dal contesto e dalla risposta.
Durante il calcolo della loss, i token appartenenti all'istruzione vengono ignorati (con una masking dell'istruzione) per focalizzare l'apprendimento solo sulla qualità della risposta.
L'obbiettivo è minimizzare l'errore della predizione data la sequenza di comando.

Ma come dobbiamo impachettare questi dati affinchè la rete li digerisca correttamente?
Sono nati diversi dialetti per i dataset
- Alpaca Style: formato classico che prevede campi separati per l'istruzione, input (opzionale) e Output. Istruzione, input, risposta.
- ChatML: diventato lo standard moderno (moderno rispetto ad alpaca style) basato su ruoli (system, user, assistant) per gestire conversazioni multi-turno complesse.
L'uso di delimitatori coerenti previene confusioni tra i diversi turni di parola e mantiene stabile lo stato interno del modello.

Dall'Autocomplete all'Assistant
Gli LLM sono un eccezionale completatore statistico di frasi. Senza Instruction tuning, se chiediamo "qual'è la capitale delle Francia?" potrebbe rispondere: "e quali sono i suoi fiumi principali? semplicemente perchè le due frasi sono state 'spesso' viste insieme.
L'SFT forza il collasso (benefico) della distribuzione probabilistica verso la risposta utile, trasformado il generatore di testo in un risolutore di problemi. Forziamo la probabilità della risposta utile ad essere immensamente più alta di quella della compilazione casuale. Stiamo chiudendo delle porte all'interno dello spazio latente in modo da lasciarne aperta una sola, qualla più utile all'utente.

Non si devono fare dataset per memorizzare dati ERP, quelli stanno nel gestionale e la duplicazione di dati porta spesso inconsistenza del dato (mancato aggiornameto da entrambe le parti)
Un esempio corretto potrebbe essere la classificazione delle mail.
Esempio 
INPUT:
"Please supply 1,500 pcs 100234
according to quotation 45872."
OUTPUT:
{
    "document_type": "sales_order",
    "lines": [
        {
            "item_reference": "100234",
            "quantity": 1500
        }
    ],
    "quotation_reference": "45872"
}

E' possibile utilizzare un formato conversazionale.
Con un modello come Qwen Instruction, puoi strutturare ogni esempio come conversazione
Esempio:
{
    "messages": [
        {
            "role": "system",
            "content": (
                "Sei un sistema ERP specializzato "
                "nell'estrazione di ordini clienti."
            )
        },
        {
            "role": "user",
            "content": """
            Buongiorno,
            ordinateci 500 pezzi ABC123.
            Consegna entro il 30 novembre.
            """
        },
        {
            "role": "assistant",
            "content": """
            {
                "document_type": "sales_order",
                "lines": [
                    {
                        "item_reference": "ABC123",
                        "quantity": 500,
                        "delivery_date": "2026-11-30"
                    }
                ]
            }
            """
        }
    ]
}

Ma attenzione, i token non vanno inventati. Usare sempre il chat template ufficiale del tokenizer del modello.
Modelli chat diversi usano formati e control token diversi, usare il template sbagliato può peggiorare pesantemente le prestazioni.

Anche qui è necessario divedere il datase tra train, validation e test.
Train: serve per modificare il modello
Validation: serve durante lo sviluppo per capire: sto migliorando? sto overfittando?, ecc
Test: deve rimanere intoccato fino alla valutazione finale.

Tutto questo è inutile se i dati che usiamo sono sporchi

Data Cleaning
Garbage In, Garbage Out
La qualità del modello finale è determinata quasi esclusivamente dalla qualità dei dati di fine-tuning, non dalla loro quantità.
1.000 esempi perfetti, valgono più di 1.000.000 di esempi mediocri.
Dataset rumorosi, contenenti informazioni sensibili o eccessivamente ripetitivi degradano drasticamente la capacità di reagionamento e la sicurezza dell'LLM
Dobbiamo pulire il dataset, ogni errore nel dataset di fine tuning verrà amplificato dal modello.

Passaggi obbligatori del data Cleaning
- Rimuovere dati duplicati o quasi identici per evitare che il modello inizi a memorizzare la risposta invece di capire la logica per generalizzare.
- Rimozione Pil (personal identify information): identificazione e mascheramento di nomi, indirizzi, e numeri di telefono per garantire la pricacy. Non vogliamo che il modello impari a memoria il numero di telefono di un cliente o l'indirizzo privato.
- Quality Filtering: utilizzo di euristiche o di modelli classifier per eliminare risposte incoerenti o grammaticalmmente errate. Usiamo modelli giudici più piccoli, per scartare le risposte scritte male o incoerenti.
Il filtraggio agisce come un setaccio che trattiene solo il segnale semantico puro necessario all'adattamento.

Come possiamo automatizzare la parte più delicata quella della pricacy?
- Uso di espressioni regolari (regex) che catturano pattern fissi come email o iban.
- Per i nomi o entità più sfumate usiamo i modelli di Named Entity Recognition per automatizzare la ricerca di dati sensibili. 
- Bilanciamento del dataset per evitare che il modello apprenda pregiudizi presenti nei testi di addestramento.
- E se non abbiamo abbastanza dati puliti? Usiamo l'ai per aiutare l'ai. Utilizzo di modelli più potenti per generare esempi di alta qualità su domini dove i dati reali sono scarsi o troppo sporchi. Esempio chiediamo a modelli enormi come GPT di scrivere esempi di alta qualità pe noi. Il modello più grande fa da mentore a quello più piccolo che sta addestrando

Analisi della Duplicazione 
Se un errore è ripetuto troppe volte nel dataset, l'LLM lo considererà una regola fondamentale dello spazio latente, smette di vederlo come errore..
Algoritmi di hashing permettono di identificare documenti duplicati in dataset di milioni di righe in tempi rapidi.

Ora che abbiamo i dati pronti, quale attrezzo usiamo per l'addestramento?

Librerie e Workflow
Strumenti per la produzione
Progettare un fine-tuning da zero è complesso.
Fortunatamente, l'ecosistema open-source offre librerie che standarizzano l'intero processo.
Utilizziamo TRL per il fine-tuning variazionale e Axoloti per la configurazione dichiarativa delle pipeline.

TRL e AXOLOTL
Standarizzazione del processo di training.
TRL (Transformer Reinforcement) di hugging face, ci mette a disposizione l'oggetto SFTTrainer per gestire automaticamente logging, checkpoint e integrazione con PEFT
E' un maggiordomo che si occupa di tutto, loggin, monitoraggio vram e salvataggio del modello 
Axolotl invece, è per chi ama l'ordine. Invece di scrivere centinaia di righe di codice, scrivi un file di configurazione YAML che semplifica l'uso simultaneo di LoRA, QLoRA e dataset diversi

Possiamo velocizzare ancora di più la l'addestramento? Si

Ottimizzazione della Pipeline
Il Packing è una tecnica che concatena più esempi brevi in un'unica sequenza per massimizzare l'uso della GPU ad ogni step. Inbvece di mangare alla GPU una frase alla volta, lasciando molto spazio nella vram, ipacchettiamo più grasi insieme in un unica sequenza lunga.
Il Gradient Accumulation (accululazione dei gradienti) permette di simulare batch size elevati sommando i gradienti di più mini-batch prima di aggiornare i pesi.

Configurazione Declarative
Riproducibilità
In un ambiente professionale, ha funzionato sul mio computer, non è una risposta accettabile, utilizzando Axolotl permette di salvare l'intera ricetta di addestramento in un singolo file di testo. Se domani devi riaddestrare un altra volta perchè sono arrivati nuovi dati, non devi ripartire da zero, carichi il file ed il sistema crea esattamente lo stesso ambiente.


In [ ]:
# piccolo fine-tuning di GPT2

# dati personalizzati -> pulizia -> formattazione prompt -> tokenizzazione 
# -> GPT-2 congelato -> LoRa trainable -> SFT 
# -> salvataggio adapter -> GPT-2 base + adapter -> generazione

# modello: Transformes
# adapter LoiRA: PEFT
# training: TRL

import torch
import re
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

# 1. DEFINIZIONE E PREPARAZIONE DEI DATI (input)
# SUPERVISED FINE-TUNING (conosci già la risposta che vuoi insegnare)
# Il dataset viene creato inizialmente come una lista di dizionari che rappresentano esempi 
# di istruzioni, contesti e risposte attese. Questa struttura è fondamentale per guidare
# il modello verso un comportamento specifico attraverso il fine-tuning supervisionato.
raw_data = [
    {"instruction": "Cos'è un tensore?", "context": "Deep Learning", "response": "Un array multidimensionale. Scrivimi a marco@tensor.it"},
    {"instruction": "Come salvo un modello?", "context": "PyTorch", "response": "Usa torch.save(). Info su docs@pytorch.org"},
    {"instruction": "Definisci il Gradiente.", "context": "Matematica", "response": "Il vettore delle derivate parziali."}
]

def preprocess_and_clean(data):
    """
    Questa funzione si occupa della pulizia dei dati per garantire la privacy e la sicurezza.
    Viene utilizzata un'espressione regolare per identificare gli indirizzi email e sostituirli
    con un testo generico [REDACTED], impedendo al modello di memorizzare dati sensibili
    durante la fase di addestramento.
    """
    email_regex = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    for entry in data:
        # La sostituzione avviene solo nel campo 'response' che contiene il testo generato.
        entry["response"] = re.sub(email_regex, "[REDACTED]", entry["response"])
    return data

# Dopo la pulizia, i dati vengono convertiti nell'oggetto Dataset della libreria Hugging Face.
# Questo oggetto permette una gestione efficiente della memoria e un'integrazione nativa
# con le classi di addestramento utilizzate successivamente.
cleaned_data = preprocess_and_clean(raw_data)
dataset = Dataset.from_list(cleaned_data)

# 2. CARICAMENTO DEL TOKENIZER
# Carichiamo il tokenizer corrispondente al modello GPT-2. Il tokenizer ha il compito di
# scomporre il testo in unità elementari (token) e convertirle in indici numerici integrabili dal modello.
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id) #carica il tokenizer corretto in base al modello scelto

# GPT-2 non dispone di un token dedicato per il padding. Per gestire batch di testi con 
# lunghezze differenti, impostiamo il token di fine sequenza (eos_token) come token di padding.
tokenizer.pad_token = tokenizer.eos_token 

# 3. CONFIGURAZIONE LoRA (Low-Rank Adaptation)
# LoRA è una tecnica di efficienza che permette di addestrare solo delle matrici aggiuntive
# a basso rango (r=8) anziché tutti i parametri del modello. Questo riduce drasticamente
# l'uso della memoria video (VRAM) pur mantenendo alte le prestazioni.
peft_config = LoraConfig(
    r=8,                       # Rango delle matrici adattive: determina la complessità dell'adattamento.
    lora_alpha=32,             # Fattore di scala che bilancia l'influenza dei pesi LoRA sui pesi originali.
    target_modules=["c_attn"], # Specifica i layer del modello GPT-2 su cui applicare le matrici LoRA.
    lora_dropout=0.05,         # Percentuale di neuroni disattivati per evitare che il modello memorizzi i dati a memoria.
    bias="none",               # Indica di non modificare i parametri di bias per massimizzare l'efficienza.
    task_type="CAUSAL_LM",     # Definisce il compito come modellazione del linguaggio casuale (generazione testo).
    fan_in_fan_out=True        # Parametro tecnico necessario per la corretta gestione dei pesi nei layer Conv1d di GPT-2.
)

# 4. DEFINIZIONE DELLA FORMATTAZIONE DEI PROMPT
def formatting_prompts_func(example):
    """
    Questa funzione riceve un singolo elemento del dataset e lo trasforma in una stringa di testo.
    La formattazione strutturata (Istruzione-Contesto-Risposta) insegna al modello a riconoscere
    il confine tra la domanda dell'utente e il corpo della risposta generata.
    """
    text = (f"### Istruzione: {example['instruction']}\n"
            f"### Contesto: {example['context']}\n"
            f"### Risposta: {example['response']}{tokenizer.eos_token}")
    return text

# 5. CONFIGURAZIONE DEL TRAINING (SFTConfig)
# SFTConfig contiene tutti i parametri che guidano il processo di addestramento.
# Questi parametri influenzano direttamente la velocità e la qualità dell'apprendimento.
sft_config = SFTConfig(
    output_dir="./gpt2-sft-trl",      # Cartella dove verranno salvati i log e i checkpoint intermedi.
    per_device_train_batch_size=1,    # Numero di esempi elaborati per ogni iterazione sulla GPU. Utile quando hai poca VRAM
    gradient_accumulation_steps=4,    # Accumula i gradienti per simulare un batch più grande e stabile.
    learning_rate=2e-4,               # Velocità con cui il modello aggiorna i propri pesi.
    logging_steps=1,                  # Intervallo di passi per mostrare l'andamento della perdita (loss).
    num_train_epochs=3,               # Numero totale di volte in cui il modello vedrà l'intero dataset.
    save_strategy="epoch",            # Salva lo stato del modello alla fine di ogni epoca di addestramento.
    fp16=True,                        # Utilizza precisione a 16 bit per velocizzare il calcolo e risparmiare VRAM.
    push_to_hub=False,                # Evita il caricamento automatico su repository remoti durante il test.
    max_length=512,                   # Lunghezza massima consentita per ogni sequenza di input.
    packing=False,                    # Mantiene ogni esempio separato per evitare interferenze tra risposte diverse.
    dataset_text_field="text"         # Identifica il campo testuale all'interno del dataset elaborato dal trainer.
)

# 6. INIZIALIZZAZIONE E AVVIO DEL PROCESSO DI TRAINING
# SFTTrainer è l'orchestratore principale: collega il modello base, il dataset preparato,
# la configurazione LoRA e i parametri di training stabiliti in SFTConfig.
trainer = SFTTrainer(
    model=model_id,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    args=sft_config,
)

# Avvio effettivo del ciclo di addestramento basato sui dati forniti.
trainer.train() # qui parte realmente l'addestramento

# 6.1 SALVATAGGIO DEFINITIVO DEGLI ADATTATORI
# Al termine del training, è fondamentale salvare i pesi addestrati (adapter) in una cartella specifica.
# Questo permette di ricaricare solo le modifiche apprese senza dover rielaborare l'intero modello.
final_adapter_path = "gpt2_lora_final"
trainer.save_model(final_adapter_path) 
print(f"Modello salvato correttamente in: {final_adapter_path}")

# 7. CARICAMENTO DEL MODELLO E TEST DI INFERENZA
# In questa fase verifichiamo che il modello abbia effettivamente appreso dai dati forniti.
# Per farlo, carichiamo il modello originale e applichiamo gli "adapter" LoRA appena creati.
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id)

# La classe PeftModel si occupa di "iniettare" gli adattatori caricati sopra il modello base.
model = PeftModel.from_pretrained(base_model, final_adapter_path)
model.to("cuda" if torch.cuda.is_available() else "cpu") # Utilizza la GPU se disponibile per la generazione.
model.eval() # Disattiva i layer di dropout e normalizzazione per una generazione coerente.

# Creiamo un prompt di prova per osservare come il modello risponde a una nuova interrogazione.
prompt = "### Istruzione: Cos'è un tensore?\n### Contesto: Deep Learning\n### Risposta:"

# Trasformiamo il prompt in tensori e chiediamo al modello di generare il completamento del testo.
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output_tokens = model.generate(
        **inputs, 
        max_new_tokens=50,       # Numero massimo di nuovi token da generare.
        temperature=0.7,         # Livello di creatività della generazione (0.7 è un buon equilibrio).
        do_sample=True,          # Abilita il campionamento probabilistico per variare le risposte.
        pad_token_id=tokenizer.eos_token_id
    )

# Decodifichiamo i token numerici prodotti dal modello in testo leggibile dall'uomo.
generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print("\n" + "="*30)
print("--- TEST INFERENZA SUCCESSIVO AL TRAINING ---")
print(generated_text)
print("="*30)

Truncating train dataset: 100%|██████████| 3/3 [00:00<00:00, 498.31 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 3/3 [00:00<00:00, 900.07 examples/s]
c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,6.200097
2,6.122725
3,6.259337


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Modello salvato correttamente in: gpt2_lora_final


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4234.53it/s]



--- TEST INFERENZA SUCCESSIVO AL TRAINING ---
### Istruzione: Cos'è un tensore?
### Contesto: Deep Learning
### Risposta: De la natale chez mazzagliare dell'información del gabeza della cata.
### Pascu: De lo que le tembla en una primera.
### H
